# Concurrencia Avanzada, Hilos y Asincronismo en Rust

Rust destaca por su "Fearless Concurrency" (concurrencia sin miedo), garantizando seguridad de memoria y ausencia de carreras de datos en tiempo de compilación. Este notebook explora desde hilos de bajo nivel hasta el modelo asíncrono moderno.

## 1. Hilos (Threads) y Estado Compartido

### A. El modelo de hilos nativos
Rust utiliza hilos 1:1 (un hilo del sistema operativo por cada hilo de Rust).

In [ ]:
use std::thread;
use std::time::Duration;

let handle = thread::spawn(|| {
    for i in 1..10 {
        println!("Hilo secundario: iteración {}", i);
        thread::sleep(Duration::from_millis(1));
    }
});

// Esperar a que el hilo termine
handle.join().expect("El hilo secundario tuvo un pánico");

### B. Comunicación mediante Canales (MPSC)
"No te comuniques compartiendo memoria; comparte memoria comunicándote".

In [ ]:
use std::sync::mpsc; // Multi-producer, single-consumer
use std::thread;

let (tx, rx) = mpsc::channel();
let tx1 = tx.clone(); // Podemos clonar el transmisor

thread::spawn(move || {
    tx.send("Mensaje desde hilo 1").unwrap();
});

thread::spawn(move || {
    tx1.send("Mensaje desde hilo 2").unwrap();
});

for recibido in rx {
    println!("Recibido: {}", recibido);
}

### C. Estado Compartido Seguro: Arc y Mutex
- `Arc<T>`: Atomic Reference Counting (para compartir propiedad entre hilos).
- `Mutex<T>`: Mutual Exclusion (para garantizar acceso exclusivo).

In [ ]:
use std::sync::{Arc, Mutex};
use std::thread;

let contador = Arc::new(Mutex::new(0));
let mut handles = vec![];

for _ in 0..10 {
    let contador = Arc::clone(&contador);
    let handle = thread::spawn(move || {
        // El lock se libera automáticamente cuando 'num' sale de ámbito
        let mut num = contador.lock().unwrap();
        *num += 1;
    });
    handles.push(handle);
}

for handle in handles { handle.join().unwrap(); }
println!("Contador final: {}", *contador.lock().unwrap());

## 2. Asincronismo (Async / Await)

A diferencia de los hilos, las tareas asíncronas son "hilos verdes" cooperativos gestionados por un **Executor** (como `tokio`).

### A. Conceptos Clave
- **Futures**: Estructuras que representan un valor que estará disponible después.
- **.await**: Suspende la ejecución actual hasta que el Future esté listo, sin bloquear el hilo del SO.
- **Runtime**: Rust no incluye un runtime asíncrono en la librería estándar (se suele usar `tokio`).

In [ ]:
// Dependencia para el notebook: :dep tokio = { version = "1", features = ["full"] }

async fn tarea_asincrona(id: u32) -> String {
    // tokio::time::sleep no bloquea el hilo
    tokio::time::sleep(std::time::Duration::from_millis(50)).await;
    format!("Tarea {} completada", id)
}

#[tokio::main]
async fn main() {
    let fut1 = tarea_asincrona(1);
    let fut2 = tarea_asincrona(2);

    // Ejecutar ambas en paralelo y esperar a ambas
    let (r1, r2) = tokio::join!(fut1, fut2);
    println!("{}, {}", r1, r2);
}

### B. Streams (Iteradores Asíncronos)
Representan una secuencia de valores que llegan a lo largo del tiempo.

In [ ]:
// use futures::stream::{self, StreamExt};
// ... procesamiento de streams con .next().await ...

## 3. Traits Send y Sync

Son la base de la seguridad en la concurrencia:
- `Send`: Permite que la propiedad del tipo se transfiera entre hilos.
- `Sync`: Permite que varios hilos accedan al tipo a través de referencias compartidas (un tipo `T` es `Sync` si `&T` es `Send`).

**Dato curioso:** Casi todos los tipos básicos son `Send` y `Sync`, excepto punteros crudos o tipos como `Rc` (que debe ser `Arc` para ser concurrente).

## 4. Barreras y Variables de Condición (Avanzado)

- `Barrier`: Permite sincronizar múltiples hilos para que todos lleguen a un punto antes de continuar.
- `Condvar`: Permite que los hilos esperen hasta que ocurra un evento específico.

In [ ]:
use std::sync::{Arc, Barrier, Mutex, Condvar};
use std::thread;

// Ejemplo de Barrera
let barrera = Arc::new(Barrier::new(3));
let mut handles = vec![];

for _ in 0..3 {
    let b = Arc::clone(&barrera);
    handles.push(thread::spawn(move || {
        println!("Esperando en la barrera...");
        b.wait();
        println!("Pasó la barrera!");
    }));
}

for h in handles { h.join().unwrap(); }